In [10]:
import subprocess
import shutil
import tempfile
import os
from pathlib import Path
import cairosvg


def svg_to_cmyk_pdf(input_svg, output_pdf, gs_path=None):
    """
    Convert SVG → CMYK PDF using CairoSVG + Ghostscript.

    Parameters
    ----------
    input_svg : str or Path
        Path to input SVG file.
    output_pdf : str or Path
        Path to output CMYK PDF.
    gs_path : str, optional
        Path to Ghostscript executable. If None, auto-detect.
    """

    input_svg = Path(input_svg)
    output_pdf = Path(output_pdf)

    if not input_svg.exists():
        raise FileNotFoundError(f"Input SVG not found: {input_svg}")

    # Locate Ghostscript
    if gs_path is None:
        gs_path = shutil.which("gs")

    if gs_path is None:
        raise RuntimeError(
            "Ghostscript ('gs') not found on PATH.\n"
            "Install it with: brew install ghostscript"
        )

    # Create temporary RGB PDF safely
    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
        temp_pdf = tmp.name

    try:
        # Step 1: SVG → RGB PDF
        cairosvg.svg2pdf(url=str(input_svg), write_to=temp_pdf)

        # Step 2: RGB → CMYK PDF via Ghostscript
        cmd = [
            gs_path,
            "-dSAFER",
            "-dBATCH",
            "-dNOPAUSE",
            "-sDEVICE=pdfwrite",
            "-sColorConversionStrategy=CMYK",
            "-dProcessColorModel=/DeviceCMYK",
            "-dUseCIEColor",
            f"-sOutputFile={output_pdf}",
        ]

        cmd.append(temp_pdf)

        subprocess.run(cmd, check=True)

        print(f"✅ CMYK PDF created: {output_pdf}")

    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Ghostscript conversion failed:\n{e}")

    finally:
        # Cleanup temp file
        if os.path.exists(temp_pdf):
            os.remove(temp_pdf)

In [11]:
# svg_to_cmyk_pdf("fig_1map.svg", "fig_1map_cmyk.pdf")

# in folder

In [12]:
import subprocess
import shutil
import tempfile
import os
from pathlib import Path
import cairosvg


def svg_to_cmyk_pdf(input_svg, output_pdf):
    gs_path = shutil.which("gs")
    if gs_path is None:
        raise RuntimeError(
            "Ghostscript ('gs') not found. Install with: brew install ghostscript"
        )

    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
        temp_pdf = tmp.name

    try:
        # Step 1: SVG → RGB PDF
        cairosvg.svg2pdf(url=str(input_svg), write_to=temp_pdf)

        # Step 2: RGB → CMYK PDF (no ICC profile)
        cmd = [
            gs_path,
            "-dSAFER",
            "-dBATCH",
            "-dNOPAUSE",
            "-sDEVICE=pdfwrite",
            "-sColorConversionStrategy=CMYK",
            "-dProcessColorModel=/DeviceCMYK",
            "-dUseCIEColor",
            f"-sOutputFile={output_pdf}",
            temp_pdf,
        ]

        subprocess.run(cmd, check=True)

    finally:
        if os.path.exists(temp_pdf):
            os.remove(temp_pdf)


def batch_convert_svg_folder(input_folder, output_folder):
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)

    if not input_folder.exists():
        raise FileNotFoundError(f"Input folder not found: {input_folder}")

    output_folder.mkdir(parents=True, exist_ok=True)

    svg_files = list(input_folder.glob("*.svg"))
    if not svg_files:
        print("No SVG files found.")
        return

    print(f"Found {len(svg_files)} SVG files.\n")

    success = 0
    failed = 0

    for svg_file in svg_files:
        output_pdf = output_folder / (svg_file.stem + ".pdf")

        try:
            svg_to_cmyk_pdf(svg_file, output_pdf)
            print(f"✅ {svg_file.name}")
            success += 1
        except Exception as e:
            print(f"❌ Failed: {svg_file.name}")
            print(f"   {e}\n")
            failed += 1

    print("\n--- Done ---")
    print(f"Successful: {success}")
    print(f"Failed: {failed}")

In [13]:
batch_convert_svg_folder("fig_svg", "fig_cmyk_pdf")

Found 4 SVG files.

GPL Ghostscript 10.06.0 (2025-09-09)
Copyright (C) 2025 Artifex Software, Inc.  All rights reserved.
This software is supplied under the GNU AGPLv3 and comes with NO WARRANTY:
see the file COPYING for details.


GPL Ghostscript 10.06.0: 

Use of -dUseCIEColor detected!
Since the release of version 9.11 of Ghostscript we recommend you do not set
-dUseCIEColor with the pdfwrite/ps2write device family.

GPL Ghostscript 10.06.0: 

Use of -dUseCIEColor detected!
Since the release of version 9.11 of Ghostscript we recommend you do not set
-dUseCIEColor with the pdfwrite/ps2write device family.



Processing pages 1 through 1.
Page 1
✅ fig_line.svg
GPL Ghostscript 10.06.0 (2025-09-09)
Copyright (C) 2025 Artifex Software, Inc.  All rights reserved.
This software is supplied under the GNU AGPLv3 and comes with NO WARRANTY:
see the file COPYING for details.
Processing pages 1 through 1.
Page 1
✅ fig_1map.svg


GPL Ghostscript 10.06.0: 

Use of -dUseCIEColor detected!
Since the release of version 9.11 of Ghostscript we recommend you do not set
-dUseCIEColor with the pdfwrite/ps2write device family.

GPL Ghostscript 10.06.0: 

Use of -dUseCIEColor detected!
Since the release of version 9.11 of Ghostscript we recommend you do not set
-dUseCIEColor with the pdfwrite/ps2write device family.



GPL Ghostscript 10.06.0 (2025-09-09)
Copyright (C) 2025 Artifex Software, Inc.  All rights reserved.
This software is supplied under the GNU AGPLv3 and comes with NO WARRANTY:
see the file COPYING for details.
Processing pages 1 through 1.
Page 1
✅ fig_3panel.svg


GPL Ghostscript 10.06.0: 

Use of -dUseCIEColor detected!
Since the release of version 9.11 of Ghostscript we recommend you do not set
-dUseCIEColor with the pdfwrite/ps2write device family.

GPL Ghostscript 10.06.0: 

Use of -dUseCIEColor detected!
Since the release of version 9.11 of Ghostscript we recommend you do not set
-dUseCIEColor with the pdfwrite/ps2write device family.



GPL Ghostscript 10.06.0 (2025-09-09)
Copyright (C) 2025 Artifex Software, Inc.  All rights reserved.
This software is supplied under the GNU AGPLv3 and comes with NO WARRANTY:
see the file COPYING for details.
Processing pages 1 through 1.
Page 1
✅ fig_4panel.svg

--- Done ---
Successful: 4
Failed: 0


GPL Ghostscript 10.06.0: 

Use of -dUseCIEColor detected!
Since the release of version 9.11 of Ghostscript we recommend you do not set
-dUseCIEColor with the pdfwrite/ps2write device family.

GPL Ghostscript 10.06.0: 

Use of -dUseCIEColor detected!
Since the release of version 9.11 of Ghostscript we recommend you do not set
-dUseCIEColor with the pdfwrite/ps2write device family.

